## FirstCry Scrapped-Data Cleaning and Preparation

## **Since this data was scraped from a structured JSON API with type casting applied while scraping only,  it looks relatively clean but standard cleaning steps are performed here still,  since they are a non-negotiable part of the analysis.**

In [55]:
# IMporting the old ususal libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Removing warnings because who wants to see them
import warnings
warnings.filterwarnings("ignore")

# Display settings for easier DataFrame viewing
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

In [34]:
df = pd.read_csv("firstcry_competitor_data.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

Shape: (978, 14)

Columns: ['product_name', 'brand', 'category', 'subcategory', 'size', 'mrp', 'discount_pct', 'selling_price', 'price_per_unit', 'stock', 'rating', 'review_count', 'source', 'search_category']

Data types:
 product_name           str
brand                  str
category               str
subcategory            str
size                   str
mrp                float64
discount_pct       float64
selling_price      float64
price_per_unit     float64
stock                int64
rating             float64
review_count         int64
source                 str
search_category        str
dtype: object


In [35]:
df.head()

,product_name,brand,category,subcategory,size,mrp,discount_pct,selling_price,price_per_unit,stock,rating,review_count,source,search_category
0,Babyhug Pro Bubble Care Premium Pant Style Dia...,Babyhug,Diapering,Baby Diapers,Medium,1449.00,53.00,677.70,12.55,17171,4.27,3330,FirstCry,diapering
1,Babyhug Advanced Pant Style Diapers Extra Larg...,Babyhug,Diapering,Baby Diapers,Extra Large,2579.00,55.00,1163.64,10.39,7902,4.10,17639,FirstCry,diapering
2,Babyhug Advanced Pant Style Diapers Large (L) ...,Babyhug,Diapering,Baby Diapers,Large,2579.00,47.00,1355.52,10.59,46584,4.10,17639,FirstCry,diapering
3,Pampers Premium Care Pants XL 72 | No Marks De...,Pampers,Diapering,Baby Diapers,Extra Large,2849.00,36.00,1819.37,25.27,887,4.48,5637,FirstCry,diapering
4,Pampers Premium Care Pants L 88 | No Marks Des...,Pampers,Diapering,Baby Diapers,Large,2849.00,39.00,1739.31,19.76,1321,4.48,5637,FirstCry,diapering


In [36]:
print("=== Null counts ===")
print(df.isnull().sum()) # no nulls were found, great

=== Null counts ===
product_name       0
brand              0
category           0
subcategory        0
size               0
mrp                0
discount_pct       0
selling_price      0
price_per_unit     0
stock              0
rating             0
review_count       0
source             0
search_category    0
dtype: int64


In [37]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

print("Columns after cleaning:", df.columns.tolist())

Columns after cleaning: ['product_name', 'brand', 'category', 'subcategory', 'size', 'mrp', 'discount_pct', 'selling_price', 'price_per_unit', 'stock', 'rating', 'review_count', 'source', 'search_category']


## **There were no null values, so we didn't need to fill any missing values, otherwise in a production-level analysis those steps would have been performed using .fillna() function, and filling the missing values with apt resutls**

In [38]:
# Making sure these columns are numeric, although they are most likely already numeric
# because we did proper data cleaning while scraping.

numeric_cols = [
    "mrp", "discount_pct", "selling_price", "price_per_unit",
    "stock", "rating", "review_count"
]

for col in numeric_cols:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(r"[^\d.]", "", regex=True)
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Null counts after numeric conversion:")
print(df[numeric_cols].isnull().sum())

Null counts after numeric conversion:
mrp               0
discount_pct      0
selling_price     0
price_per_unit    0
stock             0
rating            0
review_count      0
dtype: int64


In [39]:
# Dropping Duplicates
before = len(df)
df = df.drop_duplicates(subset=["product_name", "brand", "category", "subcategory", "size"])
print(f"Duplicates removed: {before - len(df)}")
print(f"Rows remaining: {len(df)}")

Duplicates removed: 0
Rows remaining: 978


In [40]:
# Sanity and logical checks

# selling_price > MRP??
invalid_price = df[df["selling_price"] > df["mrp"]]
print(f"Selling price > MRP: {len(invalid_price)} rows")

df.loc[df["selling_price"] > df["mrp"], "selling_price"] = df["mrp"] # We set selling price equal to MRP because selling price should not be higher than MRP.

# rating < 0 or rating > 5??
invalid_rating = df[(df["rating"] > 5) | (df["rating"] < 0)]
print(f"Invalid rating: {len(invalid_rating)} rows")


# discount < 0 or discount > 100??
invalid_disc = df[(df["discount_pct"] > 100) | (df["discount_pct"] < 0)]
print(f"Invalid discount: {len(invalid_disc)} rows")


# in source all are FirstCry so dropping that column
df = df.drop(columns=["source"], errors="ignore")

# category and search_category look very similar, so checking if they are entirely same
print((df["category"] != df["search_category"]).sum()) # sum is equal to total rows, so dropping search_category column belwo
df = df.drop(columns=["search_category"], errors="ignore")

Selling price > MRP: 0 rows
Invalid rating: 0 rows
Invalid discount: 0 rows
978


In [41]:
# saving to clean file and printing info abt the dataset
df.to_csv("firstcry_clean.csv", index=False)

print("Saved. Final shape:", df.shape)
print("\nCategory distribution:")
print(df["category"].value_counts())
print("\nSubcategory distribution:")
print(df["subcategory"].value_counts())

Saved. Final shape: (978, 12)

Category distribution:
category
Diapering    978
Name: count, dtype: int64

Subcategory distribution:
subcategory
Baby Diapers                 472
Cloth Diapers/ Nappies       171
Baby Wipes                    86
Diaper Changing Pads/Mats     81
Potty Training                79
Diaper Bag & Backpacks        74
Diaper Rash Cream             15
Name: count, dtype: int64


# FEATURE ENGINEERING

In [42]:
# Feature 1 - Demand Signal
df["demand_signal"] = df["rating"] * np.log(df["review_count"] + 1)
df.sort_values("demand_signal", ascending=False).head()

,product_name,brand,category,subcategory,size,mrp,discount_pct,selling_price,price_per_unit,stock,rating,review_count,demand_signal
50,Babyhug Advanced Pant Style Diapers Small (S) ...,Babyhug,Diapering,Baby Diapers,Small,1289.00,50.00,642.57,7.65,7969,4.10,17639,40.09
85,Babyhug Advanced Pant Style Diaper New Born (N...,Babyhug,Diapering,Baby Diapers,New Born,399.00,38.00,246.50,7.25,20005,4.10,17639,40.09
1,Babyhug Advanced Pant Style Diapers Extra Larg...,Babyhug,Diapering,Baby Diapers,Extra Large,2579.00,55.00,1163.64,10.39,7902,4.10,17639,40.09
381,Babyhug Advanced Pant Style Diapers XXL - 22 P...,Babyhug,Diapering,Baby Diapers,XXL,589.00,50.00,296.80,13.49,3400,4.10,17639,40.09
57,Babyhug Advanced Pant Style Diaper XXXL Size -...,Babyhug,Diapering,Baby Diapers,XXXL,1099.00,53.00,519.83,25.99,19003,4.10,17639,40.09


In [43]:
# Feature 1 changes
# SKU-level demand signal: rating only
# review_count is at parent product level on FirstCry, not SKU level
# using it here creates false ties across size variants of the same product
df["demand_signal"] = df["rating"]

# Brand-level demand signal: aggregated separately for brand analysis
brand_demand = (
    df.groupby(["brand", "subcategory"])
    .agg(
        avg_rating=("rating", "mean"),
        max_reviews=("review_count", "max")  # max avoids summing duplicates
    )
    .reset_index()
)
brand_demand["brand_demand_signal"] = (
    brand_demand["avg_rating"] * np.log1p(brand_demand["max_reviews"])
)

print("Top brands by demand signal:")
print(brand_demand.sort_values("brand_demand_signal", ascending=False).head(10))

Top brands by demand signal:
                 brand                subcategory  avg_rating  max_reviews  \
11             Babyhug               Baby Diapers        4.19        17639   
48             Huggies               Baby Diapers        4.24         7501   
81             Pampers               Baby Diapers        4.25         6996   
12             Babyhug                 Baby Wipes        4.28         3012   
15             Babyhug  Diaper Changing Pads/Mats        4.40         2274   
72  Mother Sparsh Baby                 Baby Wipes        4.51         1583   
63            MamyPoko               Baby Diapers        4.36         1737   
86        R for Rabbit               Baby Diapers        4.43          905   
65             Mee Mee                 Baby Wipes        4.29         1098   
42   Himalaya Babycare          Diaper Rash Cream        4.48          761   

    brand_demand_signal  
11                40.96  
48                37.82  
81                37.63  
12      

### A product flagged here has an MRP more than 50% above its subcategory median. This does not confirm deceptive pricing. It flags products where the discount percentage should be interpreted with caution, particularly if the brand is not established as a premium player.

In [44]:
# Feature 2: MRP Inflation Flag 
subcat_median_mrp = df.groupby("subcategory")["mrp"].transform("median")

df["mrp_outlier_flag"] = df["mrp"] > (subcat_median_mrp * 1.5)

print("Products with inflated MRP:")
print(df[df["mrp_outlier_flag"] == True]
      [["product_name", "brand", "subcategory", "mrp", "discount_pct"]]
      .head(10))

print(f"\nTotal flagged: {df['mrp_outlier_flag'].sum()} out of {len(df)} products")

Products with inflated MRP:
                                         product_name     brand   subcategory  \
1   Babyhug Advanced Pant Style Diapers Extra Larg...   Babyhug  Baby Diapers   
2   Babyhug Advanced Pant Style Diapers Large (L) ...   Babyhug  Baby Diapers   
3   Pampers Premium Care Pants XL 72 | No Marks De...   Pampers  Baby Diapers   
4   Pampers Premium Care Pants L 88 | No Marks Des...   Pampers  Baby Diapers   
5   Babyhug Advanced Pant Style Diapers Medium (M)...   Babyhug  Baby Diapers   
8   MamyPoko Pants Extra Absorb Baby Diapers, Extr...  MamyPoko  Baby Diapers   
9   Babyhug Advanced Pant Style Diaper Monthly Box...   Babyhug  Baby Diapers   
12  MamyPoko Pants Extra Absorb Baby Diapers, Larg...  MamyPoko  Baby Diapers   
14  Pampers Active Baby Taped Diapers, Medium size...   Pampers  Baby Diapers   
15  Pampers Premium Care Pants M 104 | No Marks De...   Pampers  Baby Diapers   

       mrp  discount_pct  
1  2579.00         55.00  
2  2579.00         47.00  

In [45]:
# Feature 3: Discount Gap 
subcat_avg_discount = df.groupby("subcategory")["discount_pct"].transform("mean")

df["discount_gap"] = df["discount_pct"] - subcat_avg_discount
# Positive = discounted more than category average
# Negative = discounted less than category average

print("Discount gap sample (most aggressive discounting):")
print(df[["product_name", "brand", "subcategory", "discount_pct", "discount_gap"]]
      .sort_values("discount_gap", ascending=False)
      .head(5))

Discount gap sample (most aggressive discounting):
                                          product_name         brand  \
475  Babyoye Air Soft Premium Pant Style Diapers XX...       Babyoye   
673  SuperBottoms Disposable Easy Clean Top Sheets ...  SuperBottoms   
118  Babyoye Air Soft Premium Pant Style Diapers Ex...       Babyoye   
725  SuperBottoms BASIC Reusable Cloth Diaper for 3...  SuperBottoms   
146  SuperBottoms BASIC Reusable Cloth Diaper for b...  SuperBottoms   

                subcategory  discount_pct  discount_gap  
475            Baby Diapers         73.00         44.10  
673  Cloth Diapers/ Nappies         73.00         41.80  
118            Baby Diapers         70.00         41.10  
725  Cloth Diapers/ Nappies         71.00         39.80  
146  Cloth Diapers/ Nappies         71.00         39.80  


In [46]:
# Feature 4
# Stock Flags 
df["stock_pct_rank"] = df.groupby("subcategory")["stock"].rank(pct=True)

df["stock_flag"] = pd.cut(
    df["stock_pct_rank"],
    bins=[0, 0.10, 0.30, 1.0],
    labels=["low", "moderate", "healthy"]
)

print("Stock flag distribution:")
print(df["stock_flag"].value_counts())


# Note: The scraper URL used OutOfStock= (empty), meaning FirstCry's API
# returned only in-stock products by default. No out-of-stock SKUs exist
# in this dataset. The stock_flag column therefore captures relative stock
# pressure within subcategory, not absolute availability.

Stock flag distribution:
stock_flag
healthy     689
moderate    197
low          92
Name: count, dtype: int64


In [47]:
# ── Feature 5: Price Per Unit Rank ────────────────────────────────────────
df["ppu_rank"] = df.groupby("subcategory")["price_per_unit"].rank(pct=True)
# 0.0 = cheapest per unit in subcategory, 1.0 = most expensive

print("Cheapest per unit by subcategory (bottom 10%):")
print(df[df["ppu_rank"] <= 0.10]
      [["product_name", "brand", "subcategory", "price_per_unit", "ppu_rank"]]
      .sort_values(["subcategory", "ppu_rank"]))

Cheapest per unit by subcategory (bottom 10%):
                                          product_name              brand  \
401  Wowper Fresh Baby Diaper Pants By DOMS | New B...             Wowper   
137  Wowper Fresh Baby Diaper Pants By DOMS | Wetne...             Wowper   
275  Babyoye Air Soft Premium Pant Style Diapers Ne...            Babyoye   
444  Wowper Fresh Baby Diaper Pants | Small (S) Siz...             Wowper   
333  Teddyy Easy Diaper Pants with Rash Protection ...             Teddyy   
622  Little's Baby Diapers L Size (Large) Pants Sty...           Little's   
361  New Mamypoko Pants Premium All Night Absorb| P...           MamyPoko   
46   Wowper Fresh Baby Diaper Pants By DOMS | Wetne...             Wowper   
889  Wowper Easy Pants Baby Diaper By DOMS |Wetness...             Wowper   
121  New Mamypoko Pants Premium All Night Absorb| P...           MamyPoko   
77   Babyhug Advanced Pant Style Diaper New Born (N...            Babyhug   
206  Teddyy Easy Diaper Pants

In [48]:
#  Brand x Subcategory Coverage feature 6
brand_subcat_coverage = (
    df.groupby(["brand", "subcategory"])
    .size()
    .reset_index(name="sku_count")
)

df = df.merge(brand_subcat_coverage, on=["brand", "subcategory"], how="left")

print("Brand coverage sample:")
print(brand_subcat_coverage.sort_values("sku_count", ascending=False).head(10))

Brand coverage sample:
            brand                subcategory  sku_count
101  SuperBottoms     Cloth Diapers/ Nappies         87
81        Pampers               Baby Diapers         85
48        Huggies               Baby Diapers         77
63       MamyPoko               Baby Diapers         65
13        Babyhug     Cloth Diapers/ Nappies         61
11        Babyhug               Baby Diapers         46
21   Bambo Nature               Baby Diapers         38
15        Babyhug  Diaper Changing Pads/Mats         30
57         LuvLap               Baby Diapers         27
86   R for Rabbit               Baby Diapers         24


In [50]:
print("Final feature columns:")
print([col for col in df.columns if col not in 
       ["product_name", "brand", "category", "subcategory", 
        "size", "mrp", "discount_pct", "selling_price", 
        "price_per_unit", "stock", "rating", "review_count", 
        "search_category"]])

print("\nShape:", df.shape)

Final feature columns:
['demand_signal', 'mrp_outlier_flag', 'discount_gap', 'stock_pct_rank', 'stock_flag', 'ppu_rank', 'sku_count']

Shape: (978, 19)


In [70]:
df.to_csv(
    'firstcry_final_data.csv',
    index=False
)